# 🔴 Solution: DoRA Linear Layer

In [ ]:
import torch
import torch.nn as nn

In [ ]:
# ✅ SOLUTION

class DoRALinear(nn.Module):
    def __init__(self, in_features: int, out_features: int, rank: int, alpha: float = 1.0):
        super().__init__()
        self.in_features = in_features
        self.out_features = out_features
        self.rank = rank
        self.alpha = alpha
        
        # Base weight V (direction)
        self.W = nn.Parameter(torch.empty(out_features, in_features))
        nn.init.kaiming_uniform_(self.W)
        
        # Magnitude vector (initialized from W's norm)
        self.m = nn.Parameter(self.W.norm(dim=1))
        
        # LoRA matrices
        self.A = nn.Parameter(torch.empty(rank, in_features))
        self.B = nn.Parameter(torch.zeros(out_features, rank))
        nn.init.kaiming_uniform_(self.A)
    
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # Compute effective weight: m * (V + BA) / ||V + BA||
        V = self.W
        BA = self.B @ self.A * (self.alpha / self.rank) if self.rank > 0 else 0
        W_eff = V + BA
        W_eff = self.m.view(-1, 1) * W_eff / W_eff.norm(dim=1, keepdim=True)
        return x @ W_eff.T

In [ ]:
# Verify
dora = DoRALinear(in_features=64, out_features=128, rank=8)
x = torch.randn(2, 10, 64)
out = dora(x)
print(f"Output shape: {out.shape}")

In [ ]:
from torch_judge import check
check("dora")